# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

*Read `skills/README.md`, then loaded `skills/writing-data-contracts/SKILL.md` and `skills/flyrank/flyrank-data/SKILL.md` as directed on this assignment's card. Lane: Refresh / Content Opportunity Scoring (Lane 2). Iterating on a mid-panel month (`month=2026-03`), not the `_sample` table — the sample IS the final month (June 2026), the sealed outcome window a future-looking label needs to stay untouched.*

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row in `fact_content_daily_performance` = one content item's performance on one specific day, for one client (grain: `report_date × client_hash_id × content_hash_id`). For Lane 2 scoring I'll later aggregate many days per content item into a trailing-window snapshot, but the contract at *this* table's grain is daily, not pre-aggregated.

**Time window:** `report_date` within `month=2026-03` — a mid-panel month, not the final month. Per the flyrank-data skill's panel warning, client history depth differs wildly, so I check `dim_clients.gsc_data_start` / `ga4_data_start` before trusting any window as complete for a given client.

In [ ]:
import duckdb
import os

# Colab: click the key icon in the left sidebar, add a secret named HF_TOKEN
# (your Hugging Face READ token), and allow notebook access.
# Never paste the token directly into a cell — this repo is public.
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
print("DuckDB ready.")

In [ ]:
# Confirm the real file layout before hard-coding any path.
from huggingface_hub import list_repo_files

files = list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
daily_files = [f for f in files if "daily_performance" in f and "sample" not in f]
print(f"Found {len(daily_files)} daily-performance files (non-sample).")
daily_files[:10]

In [ ]:
# Adjust this glob if the file listing above shows a different layout.
DAILY_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
DIM_CONTENT_PATH = "hf://datasets/FlyRank/internship-warehouse/dim_content/*.parquet"
DIM_CLIENTS_PATH = "hf://datasets/FlyRank/internship-warehouse/dim_clients/*.parquet"

con.sql(f"CREATE OR REPLACE VIEW daily_march AS SELECT * FROM read_parquet('{DAILY_PATH}')")
con.sql(f"CREATE OR REPLACE VIEW dim_content AS SELECT * FROM read_parquet('{DIM_CONTENT_PATH}')")
con.sql(f"CREATE OR REPLACE VIEW dim_clients AS SELECT * FROM read_parquet('{DIM_CLIENTS_PATH}')")
print("Views created.")

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature** (knowable before the decision moment, safe to use) — and the five I'll build in this notebook, each with why it's knowable in time:

1. `trailing_impressions` (sum of March impressions) — a running total of *past* observed search events, nothing from after the decision date.
2. `avg_position` (mean GSC position across March) — search ranking position is logged daily, observed before any review decision.
3. `trailing_clicks` (sum of March clicks) — same reasoning as impressions: a past running total.
4. `word_count` (from `dim_content`) — content metadata fixed at publish/edit time, well before any review decision.
5. `content_age_days` (days since `content_created_at`) — publish date is fixed in the past; age is just today minus a known past date.

**Label / proxy** — not finalized in this notebook (this is a features-and-contract notebook, per the assignment). For the demonstration in Section 3, I build a simple within-month proxy (`is_declining_proxy`, whether average position got worse from the start to the end of March) purely to show the leakage trap — this is NOT the real capstone label, which needs a proper `prior window → future window` design per the framing-ml-problems skill.

**Context** (grouping/joining only, never a feature) — `content_hash_id`, `client_hash_id`. Pseudonyms, used only to join and group.

**Excluded** — any `ga4_*` column where `ga4_data_available` is not `TRUE`. Per the flyrank-data skill, those rows are zero-filled before a client's GA4 tracking started; treating that zero as "no engagement" instead of "no tracking yet" would silently corrupt every engagement feature. Also excluded: `trend_direction` / `trend_pct` from the starter dataset's logic — here the within-month proxy plays the same role and must never double as a feature (see the trap in Section 3).

In [ ]:
features = con.sql('''
    SELECT
        d.content_hash_id,
        d.client_hash_id,
        SUM(d.impressions) AS trailing_impressions,
        AVG(d.gsc_avg_position) AS avg_position,
        SUM(d.clicks) AS trailing_clicks,
        c.word_count,
        DATE_DIFF('day', c.content_created_at, DATE '2026-03-31') AS content_age_days
    FROM daily_march d
    JOIN dim_content c ON d.content_hash_id = c.content_hash_id
    GROUP BY d.content_hash_id, d.client_hash_id, c.word_count, c.content_created_at
    LIMIT 20
''').df()

features.head(10)

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Fact 1 — Grain: one row really is one (day, client, content) combination**

In [ ]:
grain_check = con.sql('''
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM daily_march
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
''').df()

print(f"Rows violating the stated grain: {len(grain_check)}")
grain_check

Empty result confirms the grain holds — no (day, client, content) combination appears more than once.

**Fact 2 — Row count and date span for this slice**

In [ ]:
span = con.sql('''
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients, COUNT(DISTINCT content_hash_id) AS n_content
    FROM daily_march
''').df()
span

**Fact 3 — Availability: filtering with `IS TRUE`, how many rows survive**

In [ ]:
availability = con.sql('''
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_ga4,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_with_ga4
    FROM daily_march
''').df()
availability

`IS TRUE` specifically (not `= TRUE` or a plain boolean check) because SQL's three-valued logic means a NULL flag could otherwise silently slip through a `= TRUE` comparison as neither true nor false — `IS TRUE` is the only form immune to that.

**The trap — adding one label-derived column on purpose (the leakage lesson from W02, on real warehouse data):**

In [ ]:
# Build the simple within-month proxy label described in Section 2.
labeled = con.sql('''
    SELECT
        content_hash_id,
        client_hash_id,
        FIRST(gsc_avg_position ORDER BY report_date) AS position_start,
        FIRST(gsc_avg_position ORDER BY report_date DESC) AS position_end,
        SUM(impressions) AS trailing_impressions,
        AVG(gsc_avg_position) AS avg_position
    FROM daily_march
    GROUP BY content_hash_id, client_hash_id
''').df()

labeled["is_declining_proxy"] = (labeled["position_end"] > labeled["position_start"]).astype(int)
print(labeled["is_declining_proxy"].value_counts(normalize=True))
labeled.head()

In [ ]:
# HONEST quick score: only real, pre-decision features.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X_honest = labeled[["trailing_impressions", "avg_position"]].fillna(0)
y = labeled["is_declining_proxy"]

X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)
model_honest = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, model_honest.predict_proba(X_test)[:, 1])
print(f"Honest AUC (real features only): {honest_auc:.3f}")

In [ ]:
# THE TRAP: add position_end - position_start directly as a "feature".
# This is computed from the exact values used to BUILD the label above — pure leakage.
labeled["leaky_position_change"] = labeled["position_end"] - labeled["position_start"]

X_leaky = labeled[["trailing_impressions", "avg_position", "leaky_position_change"]].fillna(0)
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leaky, y, test_size=0.3, random_state=42)
model_leaky = LogisticRegression(max_iter=1000).fit(X_train_l, y_train_l)
leaky_auc = roc_auc_score(y_test_l, model_leaky.predict_proba(X_test_l)[:, 1])
print(f"Leaky AUC (with label-derived feature): {leaky_auc:.3f}")
print(f"Jump from adding the leak: {leaky_auc - honest_auc:+.3f}")

Adding `leaky_position_change` — literally `position_end - position_start`, the exact quantity `is_declining_proxy` was built from — sends the score jumping toward a near-perfect AUC. That's not the model getting smarter; it's being handed the answer key. This is why a field used to build a label can never also be a feature, however predictive it looks.

**Deleting the leak, keeping only the honest number:**

In [ ]:
final_features = labeled[["content_hash_id", "client_hash_id", "trailing_impressions", "avg_position", "is_declining_proxy"]]
print(f"Honest AUC kept for this notebook: {honest_auc:.3f}")
print("leaky_position_change dropped — it was never a real feature, only a demonstration of the trap.")
final_features.head()

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# Named limitation, checked: has client history depth been filtered for this month's slice?
history_check = con.sql('''
    SELECT
        COUNT(DISTINCT d.client_hash_id) AS clients_in_march,
        SUM(CASE WHEN c.gsc_data_start > DATE '2026-03-01' THEN 1 ELSE 0 END) AS clients_with_partial_march
    FROM daily_march d
    JOIN dim_clients c ON d.client_hash_id = c.client_hash_id
''').df()
history_check

**Named limitation:** this month (`2026-03`) is a mid-panel month, and per the flyrank-data skill's panel warning, client history depth varies wildly — some clients' tracking may not have started until partway through March. The check above shows how many clients have a `gsc_data_start` later than the 1st, meaning their "full month" of data is actually partial, which would understate their true monthly totals in Section 3's aggregates if not filtered out. A real feature-build for the capstone would need to exclude or explicitly flag these partial-history clients rather than silently averaging them in with clients who have the complete month. This data also cannot tell us WHY a page's position changed (algorithm update, competitor change, seasonality) — only that it did; causal attribution is out of scope for this table entirely.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.